In [6]:
# Import the needed libraries

import torch
from torch import nn
from torch.nn import functional as F
import random
import math
import re
import time
import pickle
import pandas as pd
import numpy as np
from tqdm import tqdm



import matplotlib.pyplot as plt
import datetime
import os
import zipfile
import shutil
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

device: cuda


In [7]:
MODE_NAME = "forward"
REVERSE = False          # <-- the only functional difference from LLMReverse.py
NUMBER_DIGITS = 3
BATCH_SIZE = 100
EPOCHS = 25               # subtraction needs more than the addition-only 15
LEARNING_RATE = 1e-3
NINP, NHEAD, NHID, NLAYERS = 128, 16, 64, 6
 
SEED = 42                 # MUST match LLMReverse.py -- see module docstring
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 50_000, 10_000, 12_000  # test >= 10k (Task 3)
 
random.seed(SEED)
torch.manual_seed(SEED)

### Tokeniser 

In [8]:
pad_token = "[PAD]"
eos_token = "[EOS]"
 
 
class character_level_tokenizer:
    def __init__(self):
        self.vocab = [str(x) for x in range(10)] + ["+", "-", "="] + [pad_token, eos_token]
        self.token_to_id = {v: k for k, v in enumerate(self.vocab)}
        self.id_to_token = {k: v for k, v in enumerate(self.vocab)}
        self.ntokens = len(self.vocab)
        self.pattern = f"[^{re.escape(''.join(self.vocab))}]"
 
    def clean(self, text):
        return re.sub(self.pattern, "", text)
 
    def pre_tokenization(self, text):
        return [c for c in text]
 
    def encode(self, text):
        return [self.token_to_id[c] for c in self.pre_tokenization(self.clean(text))]
 
    def decode(self, token_list):
        return "".join(self.id_to_token[i] for i in token_list)
 
 
tokenizer = character_level_tokenizer()
ntokens = tokenizer.ntokens
EQ_ID = tokenizer.token_to_id["="]
EOS_ID = tokenizer.token_to_id[eos_token]
PAD_ID = tokenizer.token_to_id[pad_token]
print(f"vocab ({ntokens} tokens): {tokenizer.vocab}")

vocab (15 tokens): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '-', '=', '[PAD]', '[EOS]']


In [10]:
def sample_datapoint(number_digits=3):
    """
    Returns a tuple containing two random numbers and their operator. 
    """
    a_list = [random.randint(0, 9) for _ in range(number_digits)]
    a_int = "".join([str(x) for x in a_list])
 
    b_list = [random.randint(0, 9) for _ in range(number_digits)]
    b_int = "".join([str(x) for x in b_list])
 
    op = random.choice(["+", "-"])
    result_int = int(a_int) + int(b_int) if op == "+" else int(a_int) - int(b_int)
 
    result_str = str(result_int)
    if REVERSE:
        target = ("-" + result_str[1:][::-1]) if result_str.startswith("-") else result_str[::-1]
    else:
        target = result_str
 
    return (str(a_int) + op + str(b_int) + "=", target)

def parse_equation(prompt):
    """Recovers (a_int, b_int, op) from a prompt like '089-957='."""
    op = "+" if "+" in prompt else "-"
    a_str, b_str = prompt.replace("=", "").split(op)
    return int(a_str), int(b_str), op
 
 
def check_carries_or_borrows(a_int, b_int, op, result_int):
    """
    Extends the workshop's check_carries to also handle subtraction (borrows),
    using the same right-to-left column walk. Takes the raw numbers directly
    (rather than a formatted data_tuple) so it works regardless of REVERSE.
    """
    number_digits = len(str(abs(result_int)))
    str_a = str(a_int).zfill(number_digits)
    str_b = str(b_int).zfill(number_digits)
 
    carry_or_borrow = 0
    used_flags = []
    for i in reversed(range(number_digits)):
        digit_a, digit_b = int(str_a[i]), int(str_b[i])
        used_flags.append(1 if carry_or_borrow == 1 else 0)
        if op == "+":
            carry_or_borrow = 1 if (digit_a + digit_b + carry_or_borrow >= 10) else 0
        else:
            carry_or_borrow = 1 if (digit_a - digit_b - carry_or_borrow < 0) else 0
    used_flags.reverse()
    return used_flags
    

### Checking the Sample data

In [11]:
sample = sample_datapoint(3)
print(sample)
a_int, b_int, op = parse_equation(sample[0])
demo_result = a_int + b_int if op == "+" else a_int - b_int
print(check_carries_or_borrows(a_int, b_int, op, demo_result))

('349+722=', '1071')
[1, 0, 1, 0]


### Setting the data parameters and creating the TVT

In [14]:
# dataset parameters
number_digits = 3
train_size = 50000
val_size = 10000
test_size = 12000 
 
data = []
while len(data) < (train_size + val_size + test_size):
    sample = sample_datapoint(number_digits)
    if sample not in data:
        data.append(sample)
 
data_train = data[:train_size]
data_val = data[train_size: train_size + val_size]
data_test = data[train_size + val_size: train_size + val_size + test_size]
 
print(len(data_train), len(data_val), len(data_test))
print(data_train[:4], data_val[:4], data_test[:4])
print(set(data_train).intersection(set(data_val)),
      set(data_train).intersection(set(data_test)),
      set(data_val).intersection(set(data_test)))

50000 10000 12000
[('549-802=', '-253'), ('629+256=', '885'), ('318+513=', '831'), ('329+569=', '898')] [('581+131=', '712'), ('818+178=', '996'), ('118+307=', '425'), ('981+606=', '1587')] [('463+485=', '948'), ('145+741=', '886'), ('639+148=', '787'), ('546+119=', '665')]
set() set() set()


### Positional Encoding 

In [17]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(1e4) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer("pe", pe)
 
    def forward(self, x):
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)
 
 
class CustomEncoderLayer(nn.TransformerEncoderLayer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.attn_weights = None
 
    def forward(self, src, src_mask=None, src_key_padding_mask=None, **kwargs):
        src2, attn_weights = self.self_attn(
            src, src, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask,
            need_weights=True, average_attn_weights=False,
        )
        self.attn_weights = attn_weights
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

### Building the model 

In [18]:
class TransformerModel(nn.Module):
    def __init__(self, ntoken, ninp, nhead, nhid, nlayers, dropout=0.5):
        super().__init__()
        self.input_emb = nn.Embedding(ntoken, ninp)
        self.pos_encoder = PositionalEncoding(ninp, dropout)
        encoder_layers = CustomEncoderLayer(ninp, nhead, nhid, 0.1)
        self.encoder = nn.TransformerEncoder(encoder_layers, nlayers)
        self.decoder = nn.Linear(ninp, ntoken)
        self.ninp = ninp
        self.init_weights()
 
    def init_weights(self):
        initrange = 0.1
        nn.init.uniform_(self.input_emb.weight, -initrange, initrange)
        nn.init.zeros_(self.decoder.bias)
        nn.init.uniform_(self.decoder.weight, -initrange, initrange)
 
    def _generate_square_subsequent_mask(self, sz):
        return torch.log(torch.tril(torch.ones(sz, sz)))
 
    def forward(self, src):
        mask = self._generate_square_subsequent_mask(len(src)).to(src.device)
        output_enc = self.input_emb(src) * math.sqrt(self.ninp)
        output_enc = self.pos_encoder(output_enc)
        output_enc = self.encoder(output_enc, mask=mask)
        output_dec = self.decoder(output_enc)
        attention_maps = [layer.attn_weights for layer in self.encoder.layers]
        return F.log_softmax(output_dec, dim=-1), output_enc, attention_maps
 
 
model = TransformerModel(ntoken=ntokens, ninp=NINP, nhead=NHEAD, nhid=NHID, nlayers=NLAYERS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"parameters: {sum(p.numel() for p in model.parameters())}")

/tmp/ipykernel_163/2482511144.py:7: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = nn.TransformerEncoder(encoder_layers, nlayers)


parameters: 502671
